# Lab 02 Solution: Multi-Step Workflow

**Goal:** Build a workflow with multiple processing steps to see how state flows through a chain of nodes.

**What you'll learn:**
- How state accumulates as it passes through nodes
- How each node can read fields set by previous nodes
- Building a realistic multi-step data processing pipeline
- Inspecting state at each stage

In [ ]:
from typing import TypedDict
from datetime import datetime
from langgraph.graph import StateGraph, START, END

## Original pipeline with TODO solutions

In [ ]:
class TicketState(TypedDict):
    raw_input: str
    cleaned_input: str
    category: str
    priority: str
    response: str
    log_entry: str  # ← TODO 2: added field

In [ ]:
def clean_input(state: TicketState) -> dict:
    cleaned = " ".join(state["raw_input"].split()).lower().strip()
    print(f"  [clean_input] '{state['raw_input']}' \u2192 '{cleaned}'")
    return {"cleaned_input": cleaned}

## TODO 1 Solution: Validation node

In [ ]:
def validate(state: TicketState) -> dict:
    """Check if cleaned_input is at least 5 characters."""
    if len(state["cleaned_input"]) < 5:
        print(f"  [validate] TOO SHORT: '{state['cleaned_input']}'")
        return {
            "category": "invalid",
            "response": "Input too short. Please provide more details.",
        }
    print(f"  [validate] OK (length: {len(state['cleaned_input'])})")
    return {}

In [ ]:
def classify(state: TicketState) -> dict:
    text = state["cleaned_input"]
    if state.get("category") == "invalid":
        print(f"  [classify] Skipping \u2014 already marked invalid")
        return {}
    if any(w in text for w in ["leave", "sick", "wfh", "vacation"]):
        category = "hr"
    elif any(w in text for w in ["deploy", "bug", "server", "database", "code"]):
        category = "tech"
    elif any(w in text for w in ["expense", "reimburse", "invoice", "bill"]):
        category = "finance"
    else:
        category = "general"
    print(f"  [classify] '{text}' \u2192 category: {category}")
    return {"category": category}

def assign_priority(state: TicketState) -> dict:
    text = state["cleaned_input"]
    if any(w in text for w in ["urgent", "down", "critical", "blocked"]):
        priority = "HIGH"
    elif any(w in text for w in ["help", "issue", "problem", "error"]):
        priority = "MEDIUM"
    else:
        priority = "LOW"
    print(f"  [assign_priority] \u2192 {priority}")
    return {"priority": priority}

def generate_response(state: TicketState) -> dict:
    if state.get("category") == "invalid":
        return {}  # Already has response from validate
    responses = {
        "hr": "Your HR request has been forwarded to the HR team.",
        "tech": "A tech support ticket has been created.",
        "finance": "Your finance query has been sent to the accounts team.",
        "general": "Your request has been logged. We'll get back to you shortly.",
    }
    base = responses.get(state["category"], responses["general"])
    response = f"[{state['priority']}] {base} (Category: {state['category']})"
    print(f"  [generate_response] \u2192 {response}")
    return {"response": response}

## TODO 2 Solution: Logging node

In [ ]:
def log_ticket(state: TicketState) -> dict:
    """Create a log entry combining all fields."""
    entry = (
        f"[{datetime.now().isoformat()}] "
        f"[{state.get('priority', 'N/A')}] "
        f"[{state.get('category', 'N/A')}] "
        f"{state.get('response', 'No response')}"
    )
    print(f"  [log] {entry}")
    return {"log_entry": entry}

## Build the complete pipeline

In [ ]:
graph = StateGraph(TicketState)
graph.add_node("clean", clean_input)
graph.add_node("validate", validate)     # ← TODO 1
graph.add_node("classify", classify)
graph.add_node("prioritize", assign_priority)
graph.add_node("respond", generate_response)
graph.add_node("log", log_ticket)        # ← TODO 2

graph.add_edge(START, "clean")
graph.add_edge("clean", "validate")      # ← TODO 1
graph.add_edge("validate", "classify")
graph.add_edge("classify", "prioritize")
graph.add_edge("prioritize", "respond")
graph.add_edge("respond", "log")         # ← TODO 2
graph.add_edge("log", END)

app = graph.compile()
print("Graph: START \u2192 clean \u2192 validate \u2192 classify \u2192 prioritize \u2192 respond \u2192 log \u2192 END")

## Test with various inputs

In [ ]:
test_tickets = [
    "I need to apply for   sick LEAVE   next week",
    "URGENT: production server is DOWN!",
    "How do I submit my   expense   report?",
    "Hi",                                # ← Too short! Tests validate
    "Where is the office cafeteria?",
]

for ticket in test_tickets:
    print(f"\nTicket: '{ticket}'")
    result = app.invoke({"raw_input": ticket})
    print(f"  Final state:")
    print(f"    category:  {result['category']}")
    print(f"    priority:  {result.get('priority', 'N/A')}")
    print(f"    response:  {result['response']}")
    print(f"    log_entry: {result.get('log_entry', 'N/A')[:60]}...")

## Key Takeaways

- TODO 1: validate node rejects inputs < 5 characters
- TODO 2: log node creates timestamped entries
- State accumulates as it passes through nodes
- Each node can read fields set by earlier nodes
- Nodes only update the fields they return